[Open in Colab](https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBotV0.ipynb)

# Moe Sticker Bot v2
Self‑host your Telegram sticker bot – import LINE / Kakao, create & manage stickers.


In [ ]:
#@title 📦 1. Setup Environment & Build Bot (Run Once)

import sys, time, subprocess, os, urllib.request, json, requests

print("✨ Ready!")

# ---------- System dependencies ----------
print("\nInstalling system packages...")
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
print("Core packages installed.")

# Go
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
print("Downloading Go...")
urllib.request.urlretrieve(url, "go.tar.gz")
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH
print(f"Go {subprocess.getoutput('go version').split()[2]} installed.")

# Python helpers
print("\nInstalling Python helpers...")
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_emoji.py -O /usr/local/bin/msb_emoji.py
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_kakao_decrypt.py -O /usr/local/bin/msb_kakao_decrypt.py
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_rlottie.py -O /usr/local/bin/msb_rlottie.py
!chmod +x /usr/local/bin/msb_emoji.py /usr/local/bin/msb_kakao_decrypt.py /usr/local/bin/msb_rlottie.py
print("Helpers installed.")

# Build the bot
print("\nBuilding MoeStickersBot...")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git 2>&1 | grep -v "Cloning"
%cd MoeStickersBot
!go mod download
!go build -o MoeStickersBot cmd/MoeStickersBot/main.go
if os.path.exists("MoeStickersBot"):
    sz = os.path.getsize("MoeStickersBot")/1024/1024
    print(f"Build complete — Binary: {sz:.1f} MB")
else:
    print("Build failed!")


In [ ]:
#@title ⚙️ 2. Configure & Launch Bot

BOT_TOKEN = ""  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]

if BOT_TOKEN:
    print(f"BOT_TOKEN = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
else:
    print("WARNING: BOT_TOKEN is missing!")

print("\nLaunching bot...")
if not BOT_TOKEN:
    print("ERROR: No BOT_TOKEN provided. Aborting.")
    sys.exit(1)

DATA_DIR = "moe_sticker_bot_data"
cmd_line = ["./MoeStickersBot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]

log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd_line, stdout=log_out, stderr=log_err)
time.sleep(3)
if process.poll() is None:
    print(f"Bot is RUNNING — PID {process.pid}")
    print("Send /start to your bot on Telegram!")
else:
    print("Bot exited immediately. Check bot_stderr.log.")
    !cat bot_stderr.log


In [ ]:
#@title 📜 3. Monitor & Control

ACTION = "View Logs"  #@param ["View Logs", "Stop Bot"]
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}

if ACTION == "View Logs":
    print(f"Last {LINES} lines of bot_{LOG_TYPE}.log:\n")
    !tail -n {LINES} bot_{LOG_TYPE}.log
else:
    print("Shutting down...")
    !pkill -f MoeStickersBot && echo "Bot terminated" || echo "No bot running"
    print("Cleanup complete.")


---
Made with ❤️ for the Sticker Community
